# FMODetect-v2 — Vast.ai resume training

Resumes from your saved Colab checkpoint (epoch 28, val −3.76) and pushes training to ~45 epochs total.

**Before you start the timer (RENT button on Vast):**
1. On your Google Drive, right-click `vot_fmo.h5` → Share → 'Anyone with the link' → copy the file ID from the URL (`https://drive.google.com/file/d/<FILE_ID>/view`).
2. Do the same for `experiments/checkpoints/run_*/best.pt`.
3. Paste both IDs into the cell below.
4. Then rent the GPU and run cells top-to-bottom.

**Hardware target:** RTX PRO 6000 96 GB ($1.20/h) → ~45 min total at batch 64.  
**Source:** https://github.com/jai-krishna-0921/FMODetect-v2

In [ ]:
#@title 0. Paste your Drive file IDs here
H5_DRIVE_ID   = "REPLACE_ME_h5_file_id"   # the 33-char ID of vot_fmo.h5
CKPT_DRIVE_ID = "REPLACE_ME_ckpt_file_id" # the 33-char ID of best.pt
RESUME_TOTAL_EPOCHS = 45                   # train until this epoch (resumes from epoch 29)
BATCH_SIZE = 64

assert H5_DRIVE_ID != "REPLACE_ME_h5_file_id", "set H5_DRIVE_ID first"
assert CKPT_DRIVE_ID != "REPLACE_ME_ckpt_file_id", "set CKPT_DRIVE_ID first"

In [ ]:
#@title 1. Verify GPU
import subprocess, torch
print(subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total,driver_version,compute_cap', '--format=csv,noheader']).decode().strip())
print('torch:', torch.__version__, '  cuda?', torch.cuda.is_available())
free, total = torch.cuda.mem_get_info()
print(f'vram free: {free//1024//1024} / {total//1024//1024} MB')

In [ ]:
#@title 2. Install deps + clone repo
import subprocess, sys, os
PKGS = ['gdown', 'scikit-image>=0.25', 'h5py>=3.12', 'opencv-python-headless>=4.10',
        'scipy>=1.14', 'tqdm>=4.66', 'imageio[ffmpeg]>=2.36', 'mlflow>=3.5',
        'pydantic>=2.9', 'pyyaml>=6.0', 'tensorboard>=2.18']
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *PKGS])

WORK = '/workspace/FMODetect-v2'
if not os.path.exists(WORK):
    subprocess.check_call(['git', 'clone', '--depth', '1', 'https://github.com/jai-krishna-0921/FMODetect-v2.git', WORK])
else:
    subprocess.check_call(['git', '-C', WORK, 'pull', '--ff-only'])
os.chdir(WORK)
print('cwd:', os.getcwd())
print(subprocess.check_output(['git', 'log', '-1', '--oneline']).decode().strip())

In [ ]:
#@title 3. Download H5 (~16 GB) + best.pt from Drive via gdown
import subprocess, os
os.makedirs('/workspace/data', exist_ok=True)
os.makedirs('/workspace/experiments/checkpoints/resume', exist_ok=True)

if not os.path.exists('/workspace/data/vot_fmo.h5'):
    print('Downloading H5 (~16 GB)...')
    subprocess.check_call(['gdown', '--id', H5_DRIVE_ID, '-O', '/workspace/data/vot_fmo.h5'])
subprocess.check_call(['ls', '-lh', '/workspace/data/vot_fmo.h5'])

if not os.path.exists('/workspace/experiments/checkpoints/resume/best.pt'):
    print('Downloading best.pt...')
    subprocess.check_call(['gdown', '--id', CKPT_DRIVE_ID, '-O', '/workspace/experiments/checkpoints/resume/best.pt'])
subprocess.check_call(['ls', '-lh', '/workspace/experiments/checkpoints/resume/best.pt'])

In [ ]:
#@title 4. Patch vast.yaml with the chosen batch + epoch count
import yaml
with open(f'{WORK}/configs/vast.yaml') as f:
    cfg = yaml.safe_load(f)
cfg['train']['batch_size'] = BATCH_SIZE
cfg['train']['epochs']     = RESUME_TOTAL_EPOCHS
with open(f'{WORK}/configs/vast.yaml', 'w') as f:
    yaml.safe_dump(cfg, f)
print(yaml.safe_dump(cfg))

In [ ]:
#@title 5. Resume training (inline so prints stream live)
import sys
sys.path.insert(0, WORK)
for m in list(sys.modules):
    if m.startswith('src.fmodetect'):
        del sys.modules[m]

from src.fmodetect.training.loop import train
from src.fmodetect.utils.config import load_config

cfg = load_config(f'{WORK}/configs/vast.yaml')
train(cfg,
      resume_from='/workspace/experiments/checkpoints/resume/best.pt',
      epochs_override=RESUME_TOTAL_EPOCHS)

In [ ]:
#@title 6. Smoke-test inference with the new checkpoint
import glob, subprocess, os, time
import numpy as np, cv2, torch
from IPython.display import Image as IPyImage, display

ckpt = sorted(glob.glob('/workspace/experiments/checkpoints/run_*/best.pt'))[-1]
print('newest ckpt:', ckpt)

from src.fmodetect.inference.runner import load_model, infer_pair
from src.fmodetect.inference.trajectory import trajectory_as_dict

model = load_model(ckpt, device='cuda')

# Pull paper's example
for n in ('ex0_im','ex0_bgr','ex1_im','ex1_bgr'):
    subprocess.run(['curl','-sL','-o',f'/tmp/{n}.png',
                    f'https://raw.githubusercontent.com/rozumden/FMODetect/master/example/{n}.png'], check=True)

out_dir = '/workspace/experiments/infer_smoke'
os.makedirs(out_dir, exist_ok=True)
for stem in ('ex0','ex1'):
    img = cv2.cvtColor(cv2.imread(f'/tmp/{stem}_im.png'),  cv2.COLOR_BGR2RGB).astype(np.float32)/255.
    bg  = cv2.cvtColor(cv2.imread(f'/tmp/{stem}_bgr.png'), cv2.COLOR_BGR2RGB).astype(np.float32)/255.
    res = infer_pair(model, img, bg, device='cuda')
    print(f'{stem}: TDF range [{res.tdf.min():.3f},{res.tdf.max():.3f}]  detections={len(res.trajectories)}')
    for i, t in enumerate(res.trajectories):
        d = trajectory_as_dict(t)
        print(f'   traj {i}: len={d["length_px"]:.1f}px speed={d["speed_px_per_frame"]:.1f}/f rad={d["radius_px"]:.1f} conf={d["confidence"]:.3f}')
    cv2.imwrite(f'{out_dir}/{stem}_overlay.png', res.overlay)
    cv2.imwrite(f'{out_dir}/{stem}_tdf.png', (np.clip(res.tdf,0,1)*255).astype(np.uint8))
    display(IPyImage(f'{out_dir}/{stem}_overlay.png'))

In [ ]:
#@title 7. Pack outputs for download
import subprocess, time
tag = time.strftime('%Y%m%d_%H%M%S')
out = f'/workspace/experiments_{tag}.tar.gz'
subprocess.check_call(['tar', 'czf', out,
                       '-C', '/workspace', 'experiments'])
subprocess.check_call(['ls', '-lh', out])
print(f'\nDownload via Vast web UI from: {out}')